In [27]:
import sys, os
import pandas as pd
from utils.misc import cols_to_front
from pathlib import Path
from datetime import datetime
import shutil

In [28]:
exp_name = "flavor_match"   # your experiment name

In [29]:
data_files = [
    'data/merged_ai_descriptors_clean.csv',
    'data/local_df_ai_descriptors.csv',
    'data/merged_ai_descriptors_dummies_filtered.csv',
    'data/merged_ai_descriptors_dummies_full.csv',
    'data/merged_ai_descriptors_clean.csv',
    'data/merged_ai_descriptors.csv',
    'data/flavor_db_descriptors.csv',
    'data/local_df_ai_descriptors.csv',
    'data/all_descriptors.csv'
]

output_files = [
    'output/flavordb_ingredients_top3_matches.xlsx',
    'output/local_ingredients_top3_matches.xlsx',
    'output/cat_clustered_jaccard.png',
    'output/db_jaccard.png',
    'output/cat_tsne.png',
    'output/db_tsne.png',
    "output/cat_jaccard_sim_top3.xlsx",
    
]

In [30]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

RUN_DIR = Path("runs") / f"{exp_name}_{timestamp}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(RUN_DIR)

runs\flavor_match_2026-02-24_12-34


In [31]:
# Save data files
data_dir = RUN_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# copy files
for file in data_files:
    src = Path(file)
    dst = data_dir / src.name
    shutil.copy2(src, dst)

In [32]:
# Save output files
data_dir = RUN_DIR / "output"
data_dir.mkdir(parents=True, exist_ok=True)

# copy files
for file in output_files:
    src = Path(file)
    dst = data_dir / src.name
    shutil.copy2(src, dst)

# Make quick cosine comparisons

In [33]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# -----------------------------
# Input files (your lists)
# -----------------------------
xlsx_files = [
    'output/local_ingredients_top3_matches.xlsx',
    'output/flavordb_ingredients_top3_matches.xlsx'
]

png_files = [
    'output/cat_tsne.png',
    'output/cat_clustered_jaccard.png',
]

out_path = RUN_DIR / "summary.png"
DPI = 300

cols_to_drop = ["raw_AI_output", "sub_category", "db", "category"]

df1 = pd.read_excel(xlsx_files[0])
df2 = pd.read_excel(xlsx_files[1])

# Drop unwanted columns
df1 = df1.drop(columns=cols_to_drop, errors="ignore")
df2 = df2.drop(columns=cols_to_drop, errors="ignore")

# Selection of rows
ings =['Macrocystis pyrifera', 'Morchella crassipes', 'Picea rubens', 'Allium canadense', 'Rhus glabra', 'Artemisia frigida']
df1 = df1[df1["Nom scientifique"].isin(ings)]   

ings = ['Rice', 'Cedar', 'Sage', 'Apple', "Celery", "Lemon"]
df2 = df2[df2["Nom scientifique"].isin(ings)]   

In [34]:


# -----------------------------
# Load PNG images
# -----------------------------
img1 = Image.open(png_files[0]).convert("RGB")
img2 = Image.open(png_files[1]).convert("RGB")

# -----------------------------
# Convert df.head() → image
# -----------------------------
def df_to_image(df, title="", dpi=300):
    df = df.astype(str)

    nrows, ncols = df.shape

    fig_w = max(12, 2.5 * ncols)
    fig_h = max(6, 1.2 * (nrows + 1))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
    ax.axis("off")

    if title:
        ax.set_title(
            title,
            fontsize=20,
            pad=20,
            fontweight="bold"
        )

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        loc="center",
        cellLoc="left"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(20)
    table.scale(1.5, 2.0)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_fontsize(22)
            cell.set_text_props(weight='bold')

    tmp = RUN_DIR / "__tmp_df.png"
    fig.savefig(tmp, bbox_inches="tight", dpi=dpi)
    plt.close(fig)

    im = Image.open(tmp).convert("RGB")
    tmp.unlink()

    return im


# Convert to images
df_img1 = df_to_image(df1, title=Path(xlsx_files[0]).name, dpi=DPI)
df_img2 = df_to_image(df2, title=Path(xlsx_files[1]).name, dpi=DPI)

# -----------------------------
# Resize all to same width
# -----------------------------
def resize_to_width(im, w):
    h = int(im.size[1] * w / im.size[0])
    return im.resize((w, h), Image.LANCZOS)

panel_w = min(
    img1.size[0], img2.size[0],
    df_img1.size[0], df_img2.size[0]
)

img1 = resize_to_width(img1, panel_w)
img2 = resize_to_width(img2, panel_w)
df_img1 = resize_to_width(df_img1, panel_w*2)
df_img2 = resize_to_width(df_img2, panel_w*2)

# -----------------------------
# Create composite (2x2 grid)
# -----------------------------
gap = 40
pad = 60
bg = (255, 255, 255)


row2_h = max(df_img1.size[1], df_img2.size[1])
row1_h = max(img1.size[1], img2.size[1])

canvas_w = pad*2 + panel_w*2 + gap
canvas_h = (
    pad*2
    + row1_h
    + gap
    + df_img1.size[1]
    + gap
    + df_img2.size[1]
)

canvas = Image.new("RGB", (canvas_w, canvas_h), bg)

# -----------------------------
# Paste images
# -----------------------------

# Top row (plots side by side)
canvas.paste(img1, (pad, pad))
canvas.paste(img2, (pad + panel_w + gap, pad))

# First dataframe (full width, centered under plots)
y_df1 = pad + row1_h + gap
canvas.paste(df_img1, (pad, y_df1))

# Second dataframe stacked below
y_df2 = y_df1 + df_img1.size[1] + gap
canvas.paste(df_img2, (pad, y_df2))
# -----------------------------
# Save at 300 DPI
# -----------------------------
out_path.parent.mkdir(exist_ok=True)
canvas.save(out_path, dpi=(DPI, DPI))

print(f"Saved composite image to: {out_path}")

Saved composite image to: runs\flavor_match_2026-02-24_12-34\composite.png


# Searching

In [19]:
kw = "iso"
flavor_db.loc[flavor_db['Nom scientifique'].str.contains(kw)]

,Nom scientifique,raw_AI_output,sub_category,db,category,descriptor
673,Bison,"gamey,earthy,lean,rich,slightly sweet,beefy,cl...",meat,flavor_db,animal product,"game,earth,lean,sweet,beef,clean,robust,grass,..."
964,Miso,"salty,umami,savory,earthy,rich,fermented,sligh...",additive,flavor_db,plant,"salt,umami,savory,earth,fermented,sweet,tang,m..."


In [20]:
flavor_db.shape

(934, 6)